<img src="./uva_seal.png">  

## Database Lock Demo

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: June 16, 2026

---  

### BACKGROUND

When multiple users can access a table in a database, there is the question of what happens if multiple users try to change the same data simultaneously.

This is particularly relevant in distributed databases.

Transactional databases must maintain data consistency and integrity, and they use a locking mechanism to prevent such changes.

When one user starts a transaction, a lock is placed on the data until it is committed. Other users cannot change the data while the lock is in place. With an *exclusive lock*, other users cannot even read, depending on the database.

Below, we see a brief demo of database locking.

---

#### Create and populate table in SQLite

In [4]:
import sqlite3

conn = sqlite3.connect("test.db")
cursor = conn.cursor()

#cursor.execute('drop table lock_test')

# create the table
cursor.execute("""
CREATE TABLE IF NOT EXISTS lock_test (
    id INTEGER PRIMARY KEY,
    value INTEGER
)
""")

# populate the table
cursor.execute("INSERT INTO lock_test VALUES (0,1)")

# retrieve the data
cursor.execute('select * from lock_test')
rows = cursor.fetchall()

for r in rows:
   print(r)
    
conn.commit()
conn.close()

(0, 1)


#### Demo Spark / SQLite database locking

1 | One Spark task opens a transaction and updates a row but does not commit, creating a lock.

2 | A second Spark task tries to update the same row. It is blocked waiting for the lock.


In [5]:
from pyspark.sql import SparkSession
import sqlite3
import time

spark = SparkSession.builder.appName("sqlite-lock-demo").master("local[*]").getOrCreate()

DB_FILE = "test.db"

def update_row(partition_id):
    conn = sqlite3.connect(DB_FILE, timeout=30)
    cursor = conn.cursor()

    print(f"Partition {partition_id} starting transaction")

    # start a transaction and immediately acquire an exclusive lock on the entire database file
    # this will increment the data on the first partition, and eventually on the db when committed
    cursor.execute("BEGIN EXCLUSIVE") 
    cursor.execute("""
        UPDATE lock_test
        SET value = value + 1
        WHERE id = 0
    """)

    print(f"Partition {partition_id} holding lock")
    time.sleep(10)   # simulate long processing

    # the lock will get released when the commit happens
    conn.commit()

    print(f"Partition {partition_id} committed")

    cursor.execute('select * from lock_test')
    rows = cursor.fetchall()

    # after the commit, we can see both updates.
    for r in rows:
        print(f'Updated value: {r}')

    conn.close()

    return [partition_id]

rdd = spark.sparkContext.parallelize([1,2], 2) # distribute partition ids 1 and 2 (workers) using 2 partitions
rdd.foreach(update_row)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 14:20:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Partition 1 starting transactionPartition 2 starting transaction    (0 + 2) / 2]

Partition 2 holding lock
Partition 2 committed
Updated value: (0, 2)
Partition 1 holding lock
Partition 1 committed=================>                             (1 + 1) / 2]
Updated value: (0, 3)
                                                                                

From the output, you should see a partition starting an update transaction and locking

When the transaction is committed, another partition can read/write

In the end, all partitions make their updates and the final data is shown
